# 07 — Standalone Accuracy Assessment, Active Learning & Multi-Epoch Trajectories

This notebook runs completely standalone on the **lean base install** of `lc-framework` (no GEE login, `luma-stack`, or ML packages required; [ADR-0005](../docs/adr/0005-standalone-modular-subpackages.md)).

It covers:
1. Olofsson et al. (2014) area-adjusted accuracy, SE, and 95% confidence intervals
2. Multi-class probability **Log-Loss** directly on a `Class Probability Stack`
3. Uncertainty-driven **Active Learning** sample suggestions (`confusion_index`, `margin`, `entropy`)
4. Multi-epoch transition matrices, **Pontius & Millones** disagreement, and HTML **Sankey** export

In [ ]:
# 1. Colab Bootstrap — lean core install (add [viz] for interactive Plotly Sankey)
# !pip install -q "git+https://github.com/<your-org>/LC_Framework.git"
# !pip install -q "lc-framework[viz] @ git+https://github.com/<your-org>/LC_Framework.git"

from pathlib import Path
import pandas as pd

from lc_framework.config import ClassDefinition, ClassSchema, ProjectConfig
from lc_framework.evaluation import (
    analyze_multi_epoch_rasters,
    compute_thematic_accuracy,
    count_raster_pixels,
    evaluate_probability_stack,
    export_sankey_html,
    suggest_active_learning_samples,
)

## 2. Load `ClassSchema` (from `ProjectConfig` or Inline)

In [ ]:
# Uncomment in Google Colab:
# from google.colab import drive
# drive.mount('/content/drive')

CONFIG_PATH = Path("../configs/example_project.yaml")
if CONFIG_PATH.exists():
    schema = ProjectConfig.from_yaml(CONFIG_PATH).class_schema
else:
    schema = ClassSchema(
        classes=[
            ClassDefinition(id=1, name="Forest", color="#1b5e20"),
            ClassDefinition(id=2, name="Agroforestry", color="#66bb6a"),
            ClassDefinition(id=3, name="Cropland", color="#fbc02d"),
            ClassDefinition(id=4, name="Built-up", color="#e53935"),
            ClassDefinition(id=5, name="Waterbody", color="#0288d1"),
        ]
    )

PROB_TIF = Path("outputs/hybrid_2025/pixel_probs_2025_hybrid_probs.tif")
VALIDATION_CSV = Path("data/validation_points.csv")  # columns: x, y, ref_class_id
REPORT_DIR = Path("outputs/evaluation_2025")

## 3. Olofsson (2014) Area-Adjusted Accuracy & Log-Loss

In [ ]:
if PROB_TIF.exists() and VALIDATION_CSV.exists():
    val_df = pd.read_csv(VALIDATION_CSV)
    coords = list(zip(val_df["x"].tolist(), val_df["y"].tolist()))
    res = evaluate_probability_stack(
        PROB_TIF,
        sample_coords_xy=coords,
        y_true=val_df["ref_class_id"].to_numpy(),
        class_schema=schema,
    )
    print("Overall Area-Weighted Accuracy:", f"{res.overall_metrics['overall_accuracy_area_weighted']:.4f}")
    print("95% CI (+/-):                  ", f"{res.overall_metrics['overall_accuracy_area_ci']:.4f}")
    print("Probability Log-Loss:          ", f"{res.overall_metrics['log_loss']:.4f}")
    res.save_reports(REPORT_DIR, prefix="eval_2025")
    display(res.class_metrics)
else:
    print("Provide a Class Probability Stack GeoTIFF and validation CSV to run evaluation.")

## 4. Active Learning Sample Suggestions
Identify spatially separated high-uncertainty pixels (`confusion_index`, `margin`, or `entropy`) for targeted re-labeling.

In [ ]:
if PROB_TIF.exists():
    suggestions = suggest_active_learning_samples(
        PROB_TIF,
        n_samples=50,
        min_distance_m=250.0,
        min_uncertainty=0.55,
        method="confusion_index",
        output_csv_path=REPORT_DIR / "active_learning_candidates.csv",
    )
    display(suggestions.head(10))

## 5. Multi-Epoch Transitions, Pontius & Millones Disagreement & Sankey Export

In [ ]:
EPOCH_MAPS = {
    "2015": Path("outputs/lulc_2015.tif"),
    "2020": Path("outputs/lulc_2020.tif"),
    "2025": Path("outputs/lulc_2025.tif"),
}

if all(p.exists() for p in EPOCH_MAPS.values()):
    traj = analyze_multi_epoch_rasters(EPOCH_MAPS, class_schema=schema)
    print("Net Disagreement (ha):", traj.net_summary.overall_disagreement)
    display(traj.net_summary.class_summary_ha)

    sankey_html = export_sankey_html(
        traj,
        REPORT_DIR / "land_cover_transitions_sankey.html",
        title="Multi-Epoch Land Cover Transitions",
        min_flow_ha=1.0,
    )
    print("Saved Sankey HTML to:", sankey_html)
else:
    print("Provide >= 2 epoch LULC GeoTIFFs in EPOCH_MAPS to run trajectory analysis.")